# 02 — 216-ROI Parcellation

This notebook extracts resting-state fMRI ROI time series for the
QC-passing paired neurofeedback cohort using the Schaefer-200 cortical
atlas and Tian-16 subcortical atlas.

The resulting 216-region time series will be used to construct Rest 1 and
Rest 2 functional connectivity matrices for brain-network reorganization
analysis.

## 1 — Inputs and Cohort

This section defines the project paths, imaging directories, atlas files,
and the QC-passing participant cohort that will be used for parcellation.

In [3]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans"
)

REST1_DIR = DATA_DIR / "Processed rest scans"
REST2_DIR = DATA_DIR / "Processed rest2 scans"

PARTICIPANTS_FILE = DATA_DIR / "participants.tsv"

print("Data folder exists:", DATA_DIR.exists())
print("Rest 1 exists:", REST1_DIR.exists())
print("Rest 2 exists:", REST2_DIR.exists())
print("Participants file exists:", PARTICIPANTS_FILE.exists())

Data folder exists: True
Rest 1 exists: True
Rest 2 exists: True
Participants file exists: True


### 1.1 QC-Passing Participant Cohort

The dataset audit identified 18 participants with complete paired Rest 1
and Rest 2 imaging data who satisfied the motion-censoring criterion in
both sessions. Only these participants are included in the parcellation
analysis.

In [5]:
QC_PASSING_IDS = [
    "E3746",
    "E3799",
    "E3973",
    "E4051",
    "E4209",
    "E4253",
    "E4324",
    "E4350",
    "E4360",
    "E4484",
    "E4689",
    "E4697",
    "E4745",
    "E5215",
    "E5580",
    "E5586",
    "E5693",
    "E5694",
]

print("QC-passing participants:", len(QC_PASSING_IDS))
print(QC_PASSING_IDS)

QC-passing participants: 18
['E3746', 'E3799', 'E3973', 'E4051', 'E4209', 'E4253', 'E4324', 'E4350', 'E4360', 'E4484', 'E4689', 'E4697', 'E4745', 'E5215', 'E5580', 'E5586', 'E5693', 'E5694']


## 2 — Atlas Verification

This section verifies the cortical and subcortical atlas files used for
ROI-based parcellation.

The analysis uses the Schaefer 200-region cortical atlas together with the
Tian 16-region subcortical atlas, giving a total of 216 ROIs.

In [7]:
from pathlib import Path

ATLAS_DIR = Path(
    r"C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code"
)

SCHAEFER_ATLAS = ATLAS_DIR / (
    "Schaefer2018_200Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
)

TIAN_ATLAS = ATLAS_DIR / (
    "Tian_Subcortex_S1_3T_2009cAsym.nii.gz"
)

print("Schaefer atlas exists:", SCHAEFER_ATLAS.exists())
print("Tian atlas exists:", TIAN_ATLAS.exists())

Schaefer atlas exists: True
Tian atlas exists: True


### 2.1 Atlas Label Count Verification

This step confirms the number of labeled regions contained in each atlas
before ROI time-series extraction.

In [9]:
import nibabel as nib
import numpy as np

schaefer_img = nib.load(str(SCHAEFER_ATLAS))
tian_img = nib.load(str(TIAN_ATLAS))

schaefer_data = schaefer_img.get_fdata()
tian_data = tian_img.get_fdata()

schaefer_labels = np.unique(schaefer_data)
tian_labels = np.unique(tian_data)

# Remove background label 0
schaefer_roi_labels = schaefer_labels[schaefer_labels != 0]
tian_roi_labels = tian_labels[tian_labels != 0]

print("Schaefer nonzero labels:", len(schaefer_roi_labels))
print("Tian nonzero labels:", len(tian_roi_labels))
print("Total ROIs:", len(schaefer_roi_labels) + len(tian_roi_labels))

print("\nSchaefer label range:",
      schaefer_roi_labels.min(),
      "to",
      schaefer_roi_labels.max())

print("Tian label range:",
      tian_roi_labels.min(),
      "to",
      tian_roi_labels.max())

Schaefer nonzero labels: 200
Tian nonzero labels: 16
Total ROIs: 216

Schaefer label range: 1.0 to 200.0
Tian label range: 1.0 to 16.0


In [10]:
import sys

print(sys.executable)

C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code\gnn_env\Scripts\python.exe


In [11]:
%pip install nibabel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code\gnn_env\Scripts\python.exe -m pip install --upgrade pip


## 3 — Single-Subject Parcellation Test

Before processing the full cohort, this section tests the parcellation
pipeline on one QC-passing participant.

The goal is to verify that the Rest 1 and Rest 2 residual fMRI datasets can
be loaded correctly and that ROI time series can be extracted using the
216-region Schaefer-Tian atlas configuration.

In [13]:
TEST_SUBJECT = "E3746"

rest1_errts = [
    p for p in REST1_DIR.rglob("*.HEAD")
    if TEST_SUBJECT in str(p)
    and "errts." in p.name.lower()
    and "rest2" not in p.name.lower()
]

rest2_errts = [
    p for p in REST2_DIR.rglob("*.HEAD")
    if TEST_SUBJECT in str(p)
    and "errts." in p.name.lower()
    and "rest2" in p.name.lower()
]

print("Rest 1 candidates:")
for p in rest1_errts:
    print(" ", p)

print("\nRest 2 candidates:")
for p in rest2_errts:
    print(" ", p)

Rest 1 candidates:
  C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest scans\E3746\errts.DP_E3746_rest.fanaticor+tlrc.HEAD

Rest 2 candidates:
  C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\DP_E3746_rest2.results\errts.DP_E3746_rest2.fanaticor+tlrc.HEAD
  C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\TP_E3746_rest2.results\errts.TP_E3746_rest2.fanaticor+tlrc.HEAD


### 3.1 Longitudinal Pair Selection

Participant E3746 contains two candidate Rest 2 processing outputs.
The Rest 1 dataset uses the `DP` prefix, so the matching longitudinal
pair is selected as `DP_E3746_rest` to `DP_E3746_rest2`.

The additional `TP_E3746_rest2` dataset is retained in the source data
but is not used for this paired analysis.

In [15]:
TEST_REST1_HEAD = next(
    p for p in rest1_errts
    if "DP_E3746_rest" in p.name
)

TEST_REST2_HEAD = next(
    p for p in rest2_errts
    if "DP_E3746_rest2" in p.name
)

print("Selected Rest 1:")
print(TEST_REST1_HEAD)

print("\nSelected Rest 2:")
print(TEST_REST2_HEAD)

Selected Rest 1:
C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest scans\E3746\errts.DP_E3746_rest.fanaticor+tlrc.HEAD

Selected Rest 2:
C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\DP_E3746_rest2.results\errts.DP_E3746_rest2.fanaticor+tlrc.HEAD


### 3.2 AFNI Dataset Loading and Shape Verification

This step loads the selected Rest 1 and Rest 2 AFNI residual datasets
for the test participant and verifies their spatial dimensions and
number of time points before parcellation.

In [17]:
rest1_img = nib.load(str(TEST_REST1_HEAD))
rest2_img = nib.load(str(TEST_REST2_HEAD))

print("Rest 1 shape:", rest1_img.shape)
print("Rest 2 shape:", rest2_img.shape)

print("\nRest 1 affine:")
print(rest1_img.affine)

print("\nRest 2 affine:")
print(rest2_img.affine)

Rest 1 shape: (96, 114, 96, 260)
Rest 2 shape: (96, 114, 96, 260)

Rest 1 affine:
[[   2.   -0.   -0.  -95.]
 [  -0.    2.   -0. -131.]
 [   0.    0.    2.  -77.]
 [   0.    0.    0.    1.]]

Rest 2 affine:
[[   2.   -0.   -0.  -95.]
 [  -0.    2.   -0. -131.]
 [   0.    0.    2.  -77.]
 [   0.    0.    0.    1.]]


### 3.3 Atlas Resampling to Functional Space

Before extracting ROI time series, the Schaefer and Tian atlases are
resampled into the spatial grid of the test participant's resting-state
fMRI image.

Nearest-neighbor interpolation is used to preserve discrete atlas labels.

In [33]:
from nilearn.image import resample_to_img

schaefer_resampled = resample_to_img(
    schaefer_img,
    rest1_img,
    interpolation="nearest"
)

tian_resampled = resample_to_img(
    tian_img,
    rest1_img,
    interpolation="nearest"
)

print("Rest 1 shape:", rest1_img.shape[:3])
print("Resampled Schaefer shape:", schaefer_resampled.shape)
print("Resampled Tian shape:", tian_resampled.shape)

Rest 1 shape: (96, 114, 96)
Resampled Schaefer shape: (96, 114, 96)
Resampled Tian shape: (96, 114, 96)


In [20]:
%pip install nilearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\Selim code\gnn_env\Scripts\python.exe -m pip install --upgrade pip


### 3.4 Resampled Atlas Label Verification

This step confirms that atlas resampling preserved the expected region
labels in functional space before ROI time-series extraction.

In [35]:
schaefer_resampled_labels = np.unique(
    schaefer_resampled.get_fdata()
)

tian_resampled_labels = np.unique(
    tian_resampled.get_fdata()
)

schaefer_resampled_rois = (
    schaefer_resampled_labels[
        schaefer_resampled_labels != 0
    ]
)

tian_resampled_rois = (
    tian_resampled_labels[
        tian_resampled_labels != 0
    ]
)

print(
    "Resampled Schaefer ROIs:",
    len(schaefer_resampled_rois)
)

print(
    "Resampled Tian ROIs:",
    len(tian_resampled_rois)
)

print(
    "Total resampled ROIs:",
    len(schaefer_resampled_rois)
    + len(tian_resampled_rois)
)

Resampled Schaefer ROIs: 200
Resampled Tian ROIs: 16
Total resampled ROIs: 216


### 3.5 Single-Subject ROI Time-Series Extraction

This step extracts cortical and subcortical ROI time series from the
test participant's Rest 1 residual fMRI data.

The Schaefer atlas provides 200 cortical ROI time series and the Tian
atlas provides 16 subcortical ROI time series, which are concatenated
to form a 216-region representation.

In [37]:
from nilearn.maskers import NiftiLabelsMasker

schaefer_masker = NiftiLabelsMasker(
    labels_img=schaefer_resampled,
    standardize=False
)

tian_masker = NiftiLabelsMasker(
    labels_img=tian_resampled,
    standardize=False
)

rest1_schaefer_ts = schaefer_masker.fit_transform(rest1_img)
rest1_tian_ts = tian_masker.fit_transform(rest1_img)

rest1_216_ts = np.concatenate(
    [rest1_schaefer_ts, rest1_tian_ts],
    axis=1
)

print("Schaefer time series shape:", rest1_schaefer_ts.shape)
print("Tian time series shape:", rest1_tian_ts.shape)
print("Combined 216-ROI shape:", rest1_216_ts.shape)

C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\2098573864.py:13: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  rest1_schaefer_ts = schaefer_masker.fit_transform(rest1_img)
C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\2098573864.py:14: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  rest1_tian_ts = tian_masker.fit_transform(rest1_img)


Schaefer time series shape: (260, 200)
Tian time series shape: (260, 16)
Combined 216-ROI shape: (260, 216)


### 3.6 Motion Censoring of ROI Time Series

The ROI time series are aligned with the AFNI motion-censor vector for
the same resting-state scan.

Censored time points are marked as missing so that motion-contaminated
volumes are not used in subsequent functional connectivity analysis.

In [39]:
# Locate E3746 Rest 1 censor file

rest1_censor_candidates = [
    p for p in REST1_DIR.rglob("*censor.1D")
    if TEST_SUBJECT in str(p)
    and "rest2" not in p.name.lower()
    and "DP_E3746_rest" in p.name
]

print("Rest 1 censor candidates:")
for p in rest1_censor_candidates:
    print(p)

Rest 1 censor candidates:
C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest scans\E3746\motion_DP_E3746_rest_censor.1D


In [41]:
REST1_CENSOR_FILE = rest1_censor_candidates[0]

rest1_censor = np.loadtxt(REST1_CENSOR_FILE)

print("Censor vector length:", len(rest1_censor))
print("Usable volumes:", int(rest1_censor.sum()))
print("Censored volumes:", int(len(rest1_censor) - rest1_censor.sum()))
print(
    "Usable fraction:",
    round(rest1_censor.sum() / len(rest1_censor), 3)
)

print(
    "Matches ROI time points:",
    len(rest1_censor) == rest1_216_ts.shape[0]
)

Censor vector length: 260
Usable volumes: 234
Censored volumes: 26
Usable fraction: 0.9
Matches ROI time points: True


In [43]:
rest1_216_ts_censored = rest1_216_ts.copy()

# AFNI censor vector:
# 1 = usable time point
# 0 = censored time point
rest1_216_ts_censored[rest1_censor == 0, :] = np.nan

print("Original time-series shape:", rest1_216_ts.shape)
print("Censored time-series shape:", rest1_216_ts_censored.shape)

print(
    "Time points marked NaN:",
    np.isnan(rest1_216_ts_censored).all(axis=1).sum()
)

print(
    "Usable time points:",
    (~np.isnan(rest1_216_ts_censored).all(axis=1)).sum()
)

Original time-series shape: (260, 216)
Censored time-series shape: (260, 216)
Time points marked NaN: 26
Usable time points: 234


### 3.7 Rest 2 ROI Extraction and Motion Censoring

The same 216-ROI extraction and motion-censoring procedure is applied
to the post-neurofeedback Rest 2 scan for the test participant.

In [45]:
# Extract Rest 2 ROI time series

rest2_schaefer_ts = schaefer_masker.fit_transform(rest2_img)
rest2_tian_ts = tian_masker.fit_transform(rest2_img)

rest2_216_ts = np.concatenate(
    [rest2_schaefer_ts, rest2_tian_ts],
    axis=1
)

print("Rest 2 combined shape:", rest2_216_ts.shape)


# Locate the matching Rest 2 censor file

rest2_censor_candidates = [
    p for p in REST2_DIR.rglob("*censor.1D")
    if TEST_SUBJECT in str(p)
    and "DP_E3746_rest2" in p.name
]

print("\nRest 2 censor candidates:")
for p in rest2_censor_candidates:
    print(p)

C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\4246802933.py:3: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  rest2_schaefer_ts = schaefer_masker.fit_transform(rest2_img)
C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\4246802933.py:4: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  rest2_tian_ts = tian_masker.fit_transform(rest2_img)


Rest 2 combined shape: (260, 216)

Rest 2 censor candidates:
C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\Processed rest2 scans\DP_E3746_rest2.results\motion_DP_E3746_rest2_censor.1D


In [47]:
REST2_CENSOR_FILE = rest2_censor_candidates[0]

rest2_censor = np.loadtxt(REST2_CENSOR_FILE)

print("Censor vector length:", len(rest2_censor))
print("Usable volumes:", int(rest2_censor.sum()))
print("Censored volumes:", int(len(rest2_censor) - rest2_censor.sum()))
print(
    "Usable fraction:",
    round(rest2_censor.sum() / len(rest2_censor), 3)
)

print(
    "Matches ROI time points:",
    len(rest2_censor) == rest2_216_ts.shape[0]
)

rest2_216_ts_censored = rest2_216_ts.copy()
rest2_216_ts_censored[rest2_censor == 0, :] = np.nan

print(
    "\nTime points marked NaN:",
    np.isnan(rest2_216_ts_censored).all(axis=1).sum()
)

print(
    "Usable time points:",
    (~np.isnan(rest2_216_ts_censored).all(axis=1)).sum()
)

Censor vector length: 260
Usable volumes: 255
Censored volumes: 5
Usable fraction: 0.981
Matches ROI time points: True

Time points marked NaN: 5
Usable time points: 255


## 4. Batch Parcellation for the QC-Passing Cohort

After validating the full Rest 1 and Rest 2 workflow on E3746, the same
216-ROI extraction and motion-censoring procedure is applied to all
18 participants who passed imaging quality control.

### 4.1 Reusable ROI Extraction Function

A reusable function applies the validated 216-ROI parcellation and
motion-censoring procedure to each resting-state scan.

In [49]:
def extract_216roi_timeseries(
    bold_img,
    schaefer_atlas,
    tian_atlas,
    censor_file
):
    # Create atlas maskers
    schaefer_masker = NiftiLabelsMasker(
        labels_img=schaefer_atlas,
        standardize=None
    )

    tian_masker = NiftiLabelsMasker(
        labels_img=tian_atlas,
        standardize=None
    )

    # Extract ROI time series
    schaefer_ts = schaefer_masker.fit_transform(bold_img)
    tian_ts = tian_masker.fit_transform(bold_img)

    # Combine 200 cortical + 16 subcortical ROIs
    ts_216 = np.concatenate(
        [schaefer_ts, tian_ts],
        axis=1
    )

    # Read AFNI censor vector
    censor = np.loadtxt(censor_file)

    if len(censor) != ts_216.shape[0]:
        raise ValueError(
            f"Censor length {len(censor)} does not match "
            f"time-series length {ts_216.shape[0]}"
        )

    # Preserve 260 TRs but mark censored TRs as NaN
    ts_216_censored = ts_216.copy()
    ts_216_censored[censor == 0, :] = np.nan

    return ts_216_censored

In [51]:
test_ts = extract_216roi_timeseries(
    rest1_img,
    schaefer_resampled,
    tian_resampled,
    REST1_CENSOR_FILE
)

print("Test output shape:", test_ts.shape)
print(
    "NaN time points:",
    np.isnan(test_ts).all(axis=1).sum()
)
print(
    "Usable time points:",
    (~np.isnan(test_ts).all(axis=1)).sum()
)

Test output shape: (260, 216)
NaN time points: 26
Usable time points: 234


### 4.2 Participant Scan Prefixes

The preprocessing prefix associated with each QC-passing participant is
specified explicitly to ensure that the correct longitudinal Rest 1 and
Rest 2 datasets are paired.

In [53]:
SCAN_PREFIXES = {
    "E3746": "DP",
    "E3799": "HL",
    "E3973": "LD",
    "E4051": "SA",
    "E4209": "LS",
    "E4253": "SA",
    "E4324": "JV",
    "E4350": "SF",
    "E4360": "KS",
    "E4484": "CB",
    "E4689": "CC",
    "E4697": "CM",
    "E4745": "LA",
    "E5215": "RS",
    "E5580": "IR",
    "E5586": "PS",
    "E5693": "KH",
    "E5694": "AC",
}

print("QC-passing participants:", len(QC_PASSING_IDS))
print("Prefix mappings:", len(SCAN_PREFIXES))

print(
    "All QC participants mapped:",
    set(QC_PASSING_IDS) == set(SCAN_PREFIXES.keys())
)

QC-passing participants: 18
Prefix mappings: 18
All QC participants mapped: True


### 4.3 File Locator for Paired Resting-State Sessions

For each QC-passing participant, the correct Rest 1 and Rest 2 AFNI
datasets and corresponding motion-censor files are identified using the
participant ID and preprocessing prefix.

In [59]:
def locate_participant_files(subject, prefix):
    # -------------------------
    # Rest 1 HEAD
    # -------------------------
    rest1_heads = [
        p for p in REST1_DIR.rglob("*.HEAD")
        if subject in str(p)
        and prefix in p.name
        and subject in p.name
        and "errts." in p.name.lower()
        and "rest2" not in p.name.lower()
    ]

    # -------------------------
    # Rest 2 HEAD
    # More flexible matching so E5693's
    # filename irregularity is handled
    # -------------------------
    rest2_heads = [
        p for p in REST2_DIR.rglob("*.HEAD")
        if subject in str(p)
        and prefix in p.name
        and subject in p.name
        and "errts." in p.name.lower()
        and "rest2" in p.name.lower()
    ]

    # -------------------------
    # Rest 1 censor
    # -------------------------
    rest1_censors = [
        p for p in REST1_DIR.rglob("*censor.1D")
        if subject in str(p)
        and prefix in p.name
        and subject in p.name
        and "rest2" not in p.name.lower()
    ]

    # -------------------------
    # Rest 2 censor
    # Flexible matching for filename variations
    # -------------------------
    rest2_censors = [
        p for p in REST2_DIR.rglob("*censor.1D")
        if subject in str(p)
        and prefix in p.name
        and subject in p.name
        and "rest2" in p.name.lower()
    ]

    # -------------------------
    # Safety checks
    # -------------------------
    if len(rest1_heads) != 1:
        raise ValueError(
            f"{subject}: expected 1 Rest1 HEAD, "
            f"found {len(rest1_heads)}"
        )

    if len(rest2_heads) != 1:
        raise ValueError(
            f"{subject}: expected 1 Rest2 HEAD, "
            f"found {len(rest2_heads)}"
        )

    if len(rest1_censors) != 1:
        raise ValueError(
            f"{subject}: expected 1 Rest1 censor, "
            f"found {len(rest1_censors)}"
        )

    if len(rest2_censors) != 1:
        raise ValueError(
            f"{subject}: expected 1 Rest2 censor, "
            f"found {len(rest2_censors)}"
        )

    return {
        "rest1_head": rest1_heads[0],
        "rest2_head": rest2_heads[0],
        "rest1_censor": rest1_censors[0],
        "rest2_censor": rest2_censors[0],
    }

In [61]:
for subject in QC_PASSING_IDS:
    prefix = SCAN_PREFIXES[subject]

    files = locate_participant_files(
        subject,
        prefix
    )

    print(subject, "OK")

E3746 OK
E3799 OK
E3973 OK
E4051 OK
E4209 OK
E4253 OK
E4324 OK
E4350 OK
E4360 OK
E4484 OK
E4689 OK
E4697 OK
E4745 OK
E5215 OK
E5580 OK
E5586 OK
E5693 OK
E5694 OK


### 4.4 Batch 216-ROI Parcellation

The validated ROI-extraction procedure is now applied to both resting-state
sessions for all 18 QC-passing participants.

In [63]:
from pathlib import Path

OUTPUT_DIR = Path("outputs") / "roi_timeseries_216"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Users\rahin\outputs\roi_timeseries_216


In [65]:
import pandas as pd
import nibabel as nib
import numpy as np
from nilearn.image import resample_to_img

roi_columns = (
    [f"Schaefer_{i:03d}" for i in range(1, 201)] +
    [f"Tian_{i:03d}" for i in range(1, 17)]
)

batch_summary = []

for subject in QC_PASSING_IDS:

    prefix = SCAN_PREFIXES[subject]

    print(f"\nProcessing {subject}...")

    # ---------------------------------
    # Locate participant files
    # ---------------------------------
    files = locate_participant_files(
        subject,
        prefix
    )

    # ---------------------------------
    # Load Rest 1 and Rest 2
    # ---------------------------------
    rest1_img_sub = nib.load(
        str(files["rest1_head"])
    )

    rest2_img_sub = nib.load(
        str(files["rest2_head"])
    )

    # ---------------------------------
    # Resample atlases to Rest 1 space
    # ---------------------------------
    schaefer_r1 = resample_to_img(
        schaefer_img,
        rest1_img_sub,
        interpolation="nearest"
    )

    tian_r1 = resample_to_img(
        tian_img,
        rest1_img_sub,
        interpolation="nearest"
    )

    # ---------------------------------
    # Resample atlases to Rest 2 space
    # ---------------------------------
    schaefer_r2 = resample_to_img(
        schaefer_img,
        rest2_img_sub,
        interpolation="nearest"
    )

    tian_r2 = resample_to_img(
        tian_img,
        rest2_img_sub,
        interpolation="nearest"
    )

    # ---------------------------------
    # Extract Rest 1 ROI time series
    # ---------------------------------
    rest1_ts = extract_216roi_timeseries(
        rest1_img_sub,
        schaefer_r1,
        tian_r1,
        files["rest1_censor"]
    )

    # ---------------------------------
    # Extract Rest 2 ROI time series
    # ---------------------------------
    rest2_ts = extract_216roi_timeseries(
        rest2_img_sub,
        schaefer_r2,
        tian_r2,
        files["rest2_censor"]
    )

    # ---------------------------------
    # Safety checks
    # ---------------------------------
    if rest1_ts.shape != (260, 216):
        raise ValueError(
            f"{subject} Rest1 unexpected shape: "
            f"{rest1_ts.shape}"
        )

    if rest2_ts.shape != (260, 216):
        raise ValueError(
            f"{subject} Rest2 unexpected shape: "
            f"{rest2_ts.shape}"
        )

    # ---------------------------------
    # Convert to DataFrames
    # ---------------------------------
    rest1_df = pd.DataFrame(
        rest1_ts,
        columns=roi_columns
    )

    rest2_df = pd.DataFrame(
        rest2_ts,
        columns=roi_columns
    )

    # ---------------------------------
    # Save CSV files
    # ---------------------------------
    rest1_output = (
        OUTPUT_DIR /
        f"{subject}_Rest1_216ROI.csv"
    )

    rest2_output = (
        OUTPUT_DIR /
        f"{subject}_Rest2_216ROI.csv"
    )

    rest1_df.to_csv(
        rest1_output,
        index=False
    )

    rest2_df.to_csv(
        rest2_output,
        index=False
    )

    # ---------------------------------
    # Count usable TRs
    # ---------------------------------
    rest1_usable = (
        ~np.isnan(rest1_ts).all(axis=1)
    ).sum()

    rest2_usable = (
        ~np.isnan(rest2_ts).all(axis=1)
    ).sum()

    batch_summary.append({
        "Subject": subject,
        "Rest1_Usable_TRs": int(rest1_usable),
        "Rest2_Usable_TRs": int(rest2_usable),
        "Rest1_Shape": str(rest1_ts.shape),
        "Rest2_Shape": str(rest2_ts.shape),
    })

    print(
        f"{subject} complete | "
        f"Rest1 usable={rest1_usable} | "
        f"Rest2 usable={rest2_usable}"
    )


print("\nBATCH PROCESSING COMPLETE")
print("Participants processed:", len(batch_summary))
print(
    "Session files expected:",
    len(batch_summary) * 2
)


Processing E3746...
E3746 complete | Rest1 usable=234 | Rest2 usable=255

Processing E3799...
E3799 complete | Rest1 usable=227 | Rest2 usable=252

Processing E3973...
E3973 complete | Rest1 usable=251 | Rest2 usable=248

Processing E4051...
E4051 complete | Rest1 usable=260 | Rest2 usable=260

Processing E4209...


C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\379675424.py:19: UserWarning: Non-finite values detected. These values will be replaced with zeros.
  schaefer_ts = schaefer_masker.fit_transform(bold_img)
C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\379675424.py:20: UserWarning: Non-finite values detected. These values will be replaced with zeros.
  tian_ts = tian_masker.fit_transform(bold_img)


E4209 complete | Rest1 usable=253 | Rest2 usable=258

Processing E4253...
E4253 complete | Rest1 usable=251 | Rest2 usable=248

Processing E4324...
E4324 complete | Rest1 usable=260 | Rest2 usable=255

Processing E4350...
E4350 complete | Rest1 usable=255 | Rest2 usable=254

Processing E4360...
E4360 complete | Rest1 usable=251 | Rest2 usable=260

Processing E4484...


C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\379675424.py:19: UserWarning: Non-finite values detected. These values will be replaced with zeros.
  schaefer_ts = schaefer_masker.fit_transform(bold_img)
C:\Users\rahin\AppData\Local\Temp\ipykernel_4800\379675424.py:20: UserWarning: Non-finite values detected. These values will be replaced with zeros.
  tian_ts = tian_masker.fit_transform(bold_img)


E4484 complete | Rest1 usable=246 | Rest2 usable=254

Processing E4689...
E4689 complete | Rest1 usable=219 | Rest2 usable=234

Processing E4697...
E4697 complete | Rest1 usable=260 | Rest2 usable=251

Processing E4745...
E4745 complete | Rest1 usable=254 | Rest2 usable=251

Processing E5215...
E5215 complete | Rest1 usable=260 | Rest2 usable=260

Processing E5580...
E5580 complete | Rest1 usable=260 | Rest2 usable=250

Processing E5586...
E5586 complete | Rest1 usable=260 | Rest2 usable=254

Processing E5693...
E5693 complete | Rest1 usable=241 | Rest2 usable=260

Processing E5694...
E5694 complete | Rest1 usable=258 | Rest2 usable=256

BATCH PROCESSING COMPLETE
Participants processed: 18
Session files expected: 36


In [67]:
# Verify saved ROI time-series files

saved_files = sorted(OUTPUT_DIR.glob("*_216ROI.csv"))

print("Total saved files:", len(saved_files))

verification = []

for file in saved_files:
    df = pd.read_csv(file)

    verification.append({
        "File": file.name,
        "Rows": df.shape[0],
        "ROIs": df.shape[1],
        "All_NaN_Rows": int(df.isna().all(axis=1).sum())
    })

verification_df = pd.DataFrame(verification)

print("\nShape counts:")
print(
    verification_df[
        ["Rows", "ROIs"]
    ].value_counts()
)

print("\nFiles with unexpected dimensions:")
print(
    verification_df[
        (verification_df["Rows"] != 260) |
        (verification_df["ROIs"] != 216)
    ]
)

verification_df.head()

Total saved files: 36

Shape counts:
Rows  ROIs
260   216     36
Name: count, dtype: int64

Files with unexpected dimensions:
Empty DataFrame
Columns: [File, Rows, ROIs, All_NaN_Rows]
Index: []


,File,Rows,ROIs,All_NaN_Rows
0,E3746_Rest1_216ROI.csv,260,216,26
1,E3746_Rest2_216ROI.csv,260,216,5
2,E3799_Rest1_216ROI.csv,260,216,33
3,E3799_Rest2_216ROI.csv,260,216,8
4,E3973_Rest1_216ROI.csv,260,216,9
